In [1]:
import sys
sys.path.append('/Users/who/projects/task1') # /path/to/directory

# Step 1: Upload the Fine-Tuned Model to Hugging Face

## 1) Prepare Model Files:
Consolidate your model files into a structure Hugging Face understands:

In [ ]:
"""
my_fine_tuned_e5/
├── adapter
│   ├── adapter_config.json
│   ├── adapter_model.safetensors
├── base_model
│   ├── config.json
│   ├── model.safetensors
├── tokenizer_config.json
├── tokenizer.json
├── special_tokens_map.json
├── vocab.txt
"""

## 2) Create a Repository on Hugging Face:
Use the Hugging Face `huggingface_hub` library to create and push the model:

In [ ]:
from huggingface_hub import HfApi, HfFolder, Repository

# Authenticate with Hugging Face
HfFolder.save_token("") #TODO: remove the token before you commit the code
api = HfApi()

# Create a repository
repo_name = "fine-tuned-e5-large-v2-lora"
repo_url = api.create_repo(repo_id=repo_name, private=True)

print(f"Repository created: {repo_url}")

Repository created: https://huggingface.co/RafaelMoveoHLS/fine-tuned-e5-large-v2-lora


## 3) Push Model Files to Hugging Face:
Use the Hugging Face repository to push your model:

In [ ]:
from huggingface_hub import upload_folder

# Local path to your model
model_directory = "fine_tuned_e5_large_v2_lora"

# Upload the model folder
upload_folder(
    folder_path=model_directory,
    repo_id="RafaelMoveoHLS/fine-tuned-e5-large-v2-lora",
    token="", #TODO: remove the token before you commit the code
    commit_message="Upload fine-tuned E5 model"
)

print(f"Model uploaded to: {repo_url}")

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/4.74M [00:00<?, ?B/s]

Model uploaded to: https://huggingface.co/fine-tuned-e5-large-v2-lora


# Step 2: Deploy the Model on SageMaker

## 1) Set Up SageMaker Environment:
Ensure you have an AWS role that SageMaker can assume. Attach these permissions:
* `AmazonSageMakerFullAccess`
* `AmazonS3FullAccess`

Install the SageMaker SDK if not already installed:
* `pip install sagemaker`

## 2) Save the Fully Combined Model (base model + LoRA adapter)

In [5]:
from transformers import AutoModel, AutoTokenizer
from peft import PeftModel, PeftConfig
import os

# Paths for loading
base_model_directory = os.path.abspath("../models/fine_tuned_e5_large_v2_lora/base_model")
adapter_directory = os.path.abspath("../models/fine_tuned_e5_large_v2_lora/adapter")
tokenizer_directory = os.path.abspath("../models/fine_tuned_e5_large_v2_lora")

# Path for saving the combined model
combined_model_directory = os.path.abspath("../models/fine_tuned_e5_large_v2_lora_combined")
os.makedirs(combined_model_directory, exist_ok=True)

# Verify that required files are present in the base model directory
assert "config.json" in os.listdir(base_model_directory), "config.json is missing in the base model directory."
assert "model.safetensors" in os.listdir(base_model_directory), "model.safetensors is missing in the base model directory."

# Load the base model, ensuring it looks only in the local directory
print("Loading base model...")
base_model = AutoModel.from_pretrained(base_model_directory, local_files_only=True)

# Load the adapter and apply it to the base model
print("Loading LoRA adapter...")
adapter_config = PeftConfig.from_pretrained(adapter_directory, local_files_only=True)
model = PeftModel.from_pretrained(base_model, adapter_directory, local_files_only=True)

# Load the tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(tokenizer_directory, local_files_only=True)

# Save the combined model (base model + LoRA adapter) into a single directory
# Save the combined model weights
# Convert the PeftModel back into a standard model and save
model = model.merge_and_unload()  # Merge adapter weights into the base model

# Save the model and tokenizer to the combined directory
model.save_pretrained(combined_model_directory)
tokenizer.save_pretrained(combined_model_directory)

print(f"Combined model (base + adapter) saved to {combined_model_directory}")
print("Tokenizer saved successfully.")


Loading base model...
Loading LoRA adapter...
Loading tokenizer...
Combined model (base + adapter) saved to /Users/who/projects/task1/models/fine_tuned_e5_large_v2_lora_combined
Tokenizer saved successfully.


### Push the Model Directory to Hugging Face

In [ ]:
from huggingface_hub import upload_folder

upload_folder(
    folder_path="../models/fine_tuned_e5_large_v2_lora_combined",
    repo_id="RafaelMoveoHLS/fine-tuned-e5-large-v2-lora",
    commit_message="Upload model with safetensors weights",
    token=""
)

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/RafaelMoveoHLS/fine-tuned-e5-large-v2-lora/commit/ad900cec0b46bac56d2c208344cd79b7545eccec', commit_message='Upload model with safetensors weights', commit_description='', oid='ad900cec0b46bac56d2c208344cd79b7545eccec', pr_url=None, repo_url=RepoUrl('https://huggingface.co/RafaelMoveoHLS/fine-tuned-e5-large-v2-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='RafaelMoveoHLS/fine-tuned-e5-large-v2-lora'), pr_revision=None, pr_num=None)

### Deploy the Model to SageMaker

In [ ]:
from sagemaker import Session
from sagemaker.huggingface import HuggingFaceModel

# Initialize a SageMaker session with a region
sagemaker_session = Session()
sagemaker_session.config = {"region_name": "eu-north-1"}  # Replace with your region

# Hugging Face Hub information
hub = {
    "HF_MODEL_ID": "RafaelMoveoHLS/fine-tuned-e5-large-v2-lora",  # Your Hugging Face repo name
    "HF_TASK": "feature-extraction",  # vector representations of text
    "HF_API_TOKEN": "",  # Your Hugging Face token
    "HF_ADAPTER": "peft==0.13.2"
}

# SageMaker IAM role
role = "" 

# Hugging Face model deployment
huggingface_model = HuggingFaceModel(
    name="skill-bbc-news-finetuned-e5-embedding-model",
    transformers_version="4.28.1",  # Replace with desired version
    pytorch_version="2.0.0",      # Replace with desired version
    py_version="py310",
    env=hub,
    role=role,
    sagemaker_session=sagemaker_session
)

# Deploy model
predictor = huggingface_model.deploy(
    initial_instance_count=1,
    endpoint_name="skill-bcc-news-finetuned-e5-embedding-endpoint",
    instance_type="ml.m5.large"  # Replace with desired instance type
)

print(f"Model deployed. Endpoint name: {predictor.endpoint_name}")

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/who/Library/Application Support/sagemaker/config.yaml
------------!Model deployed. Endpoint name: skill-bcc-news-finetuned-e5-embedding-endpoint


In [3]:
response = predictor.predict({"inputs": "New embedding generation"})
print(response[0][0])

[-0.16196578741073608, -1.2399696111679077, 0.76059889793396, -0.33956873416900635, -1.248421311378479, 0.512719452381134, -1.2094544172286987, -0.7242961525917053, -0.9036394357681274, 0.18843458592891693, 0.763290524482727, -0.690678596496582, 0.6218171715736389, 1.0227551460266113, 0.06348704546689987, 0.5047379732131958, -0.557056725025177, -0.005114376544952393, 0.43820199370384216, -1.1640182733535767, 0.22603893280029297, -1.191429615020752, -0.5216776132583618, 0.3045090138912201, -0.3208080530166626, -0.6121516227722168, -0.3605630099773407, -0.6290279030799866, -0.509661853313446, 0.4512692987918854, 0.20862434804439545, 0.21943753957748413, 0.1897774487733841, -0.6484593152999878, 1.0903903245925903, 0.00011343205551384017, -1.0713728666305542, -1.4832801818847656, 0.5495734214782715, 0.3650408685207367, -0.40140432119369507, 0.853277862071991, -1.1833313703536987, 0.9179753661155701, 0.7827714681625366, -0.06415712088346481, -0.7300678491592407, -0.17761263251304626, 0.0401

In [6]:
import boto3
import json

# Initialize the SageMaker runtime client
sagemaker_runtime = boto3.client(
    "sagemaker-runtime",
    region_name="eu-north-1"
)

# Input text to be embedded
input_text = "Test query for embedding generation"

# Payload
payload = {"inputs": input_text}

# Send the request to the SageMaker endpoint
response = sagemaker_runtime.invoke_endpoint(
    EndpointName="skill-bcc-news-finetuned-e5-embedding-endpoint",
    ContentType="application/json",
    Body=json.dumps(payload)
)

# Parse the response
response_body = json.loads(response["Body"].read().decode("utf-8"))
print("Embedding:", response_body[0][0])


Embedding: [-0.14969341456890106, -1.3180339336395264, 0.5151588916778564, -0.0594966746866703, -1.3995927572250366, 0.4392540752887726, -1.2272738218307495, -0.2039681226015091, -0.9442525506019592, 0.306510329246521, 0.721598207950592, -0.49057987332344055, 0.7192069888114929, 0.7516257166862488, -0.2660864591598511, 0.576158881187439, -0.46072596311569214, 0.38905513286590576, 0.4971175789833069, -0.8778526186943054, 0.30860233306884766, -0.6495194435119629, -0.5664334893226624, 0.007575258146971464, -0.26544836163520813, -0.758786141872406, -0.39992356300354004, -1.0235638618469238, -0.501196026802063, 0.6799569129943848, 0.054989978671073914, -0.10145040601491928, 0.32919785380363464, -0.4390730559825897, 0.8429540395736694, 0.10885539650917053, -1.2575392723083496, -1.4275015592575073, 0.5693051218986511, 0.34094807505607605, 0.13870947062969208, 0.6266096830368042, -0.9833128452301025, 1.0279552936553955, 0.5389821529388428, -0.04311709105968475, -0.9609618186950684, -0.01343677